In [1]:
import pandas as pd
import re
import os

In [14]:
print(os.listdir(".."))

['.git', '.gitattributes', '.gitignore', 'allsides_8values.csv', 'app.py', 'BABE_combined.csv', 'Backend', 'data', 'notebooks', 'old_bias_text.csv', 'README.md', 'requirements.txt', 'stereoset_intersentence.csv', 'wino_bias_full.csv']


In [3]:
babe_df = pd.read_csv("../BABE_combined.csv")

allsides_df = pd.read_csv("../allsides_8values.csv")

wino_df = pd.read_csv("../wino_bias_full.csv")

old_bias_df = pd.read_csv("../old_bias_text.csv")

stereo_df = pd.read_csv("../stereoset_intersentence.csv")

In [29]:
print("BABE:")
print(babe_df.columns)

print("\nAllSides:")
print(allsides_df.columns)

print("\nWinoBias:")
print(wino_df.columns)

print("\nOld Bias:")
print(old_bias_df.columns)

print("\nStereoSet:")
print(stereo_df.columns)

BABE:
Index(['text', 'label'], dtype='object')

AllSides:
Index(['text', 'label'], dtype='object')

WinoBias:
Index(['text', 'label'], dtype='object')

Old Bias:
Index(['text', 'label'], dtype='object')

StereoSet:
Index(['text', 'label'], dtype='object')


In [27]:
# FORMAT BABE

babe_df = babe_df[['text', 'label']]

print(babe_df.head())

print(babe_df.shape)

                                                text  label
0  As the Black Lives Matter movement grows, comp...      0
1  The case of Rahaf Mohammed al-Qunun drawn new ...      0
2  The Post said the talks on payroll taxes were ...      0
3  Nearly 78 percent of Americans report experien...      0
4  Colin P. Clarke has been teaching a course on ...      0
(4121, 2)


In [31]:
# FORMAT WINOBIAS

wino_df = wino_df[['tokens']]

wino_df.columns = ['text']

wino_df['label'] = 1

print(wino_df.head())

print(wino_df.shape)

KeyError: "None of [Index(['tokens'], dtype='object')] are in the [columns]"

In [30]:
# FORMAT ALLSIDES
# RUN THIS ONLY ONCE

allsides_df['text'] = (
    allsides_df['Root_Topic'].astype(str)
    + " news about " +
    allsides_df['Event'].astype(str)
)

allsides_df['label'] = 1

allsides_df = allsides_df[['text', 'label']]

print(allsides_df.head())

print(allsides_df.shape)

KeyError: 'Root_Topic'

In [32]:
old_bias_df = old_bias_df[['Text', 'Label']]

old_bias_df.columns = ['text', 'label']

old_bias_df['label'] = old_bias_df['label'].apply(
    lambda x: 1 if str(x).lower() == 'stereo' else 0
)

old_bias_df.head()

KeyError: "None of [Index(['Text', 'Label'], dtype='object')] are in the [columns]"

In [33]:
stereo_df = stereo_df[['context']]

stereo_df.columns = ['text']

stereo_df['label'] = 1

stereo_df.head()

KeyError: "None of [Index(['context'], dtype='object')] are in the [columns]"

In [34]:
print(babe_df.head())

print(allsides_df.head())

print(wino_df.head())

print(old_bias_df.head())

print(stereo_df.head())

                                                text  label
0  As the Black Lives Matter movement grows, comp...      0
1  The case of Rahaf Mohammed al-Qunun drawn new ...      0
2  The Post said the talks on payroll taxes were ...      0
3  Nearly 78 percent of Americans report experien...      0
4  Colin P. Clarke has been teaching a course on ...      0
                                                text  label
0  Social news about Google Holds Illegal Monopol...      1
1  Social news about Alleged 4chan Hack Exposes E...      1
2  Social news about Chinese Electric Vehicle Com...      1
3  Social news about Mark Rober’s Tesla Drove Its...      1
4  Social news about Hinge and Tinder to Roll Out...      1
                                                text  label
0  ['The' 'janitor' 'reprimanded' 'the' 'accounta...      1
1  ['The' 'carpenter' 'always' 'ask' 'the' 'libra...      1
2  ['The' 'carpenter' 'always' 'asks' 'the' 'libr...      1
3  ['The' 'physician' 'wanted' 'to' 'mee

In [35]:
df = pd.concat(
    [
        babe_df,
        allsides_df,
        wino_df,
        old_bias_df,
        stereo_df
    ],
    ignore_index=True
)

print(df.shape)

df.head()

(9241, 2)


,text,label
0,"As the Black Lives Matter movement grows, comp...",0
1,The case of Rahaf Mohammed al-Qunun drawn new ...,0
2,The Post said the talks on payroll taxes were ...,0
3,Nearly 78 percent of Americans report experien...,0
4,Colin P. Clarke has been teaching a course on ...,0


In [36]:
print("Dataset shape:", df.shape)

Dataset shape: (9241, 2)


In [37]:
df.dropna(inplace=True)

In [38]:
# CLEAN TEXT

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    return text


df['text'] = df['text'].apply(clean_text)

df.head()

,text,label
0,as the black lives matter movement grows compa...,0
1,the case of rahaf mohammed alqunun drawn new g...,0
2,the post said the talks on payroll taxes were ...,0
3,nearly percent of americans report experienci...,0
4,colin p clarke has been teaching a course on t...,0


In [39]:
# REMOVE DUPLICATES

before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print("Rows before:", before)

print("Rows after:", after)

print("Duplicates removed:", before - after)

Rows before: 9241
Rows after: 9221
Duplicates removed: 20


In [41]:
# CHECK LABEL BALANCE

df['label'].value_counts()

label
1    2265
0    2265
Name: count, dtype: int64

In [42]:
# BALANCE DATASET

biased = df[df['label'] == 1]

neutral = df[df['label'] == 0]

min_size = min(len(biased), len(neutral))


biased = biased.sample(
    n=min_size,
    random_state=42
)

neutral = neutral.sample(
    n=min_size,
    random_state=42
)


df = pd.concat(
    [biased, neutral]
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

df['label'].value_counts()

label
0    2265
1    2265
Name: count, dtype: int64

In [43]:
# FINAL DATASET SIZE

print(df.shape)

(4530, 2)


In [44]:
# TRAIN TEST SPLIT

train = df.sample(
    frac=0.8,
    random_state=42
)

test = df.drop(train.index)

print("Train shape:", train.shape)

print("Test shape:", test.shape)

Train shape: (3624, 2)
Test shape: (906, 2)


In [45]:
# SAVE FINAL CSV

df.to_csv(
    "../data/data.csv",
    index=False
)

In [46]:
# VERIFY SAVE

saved_df = pd.read_csv("../data/data.csv")

saved_df.head()

,text,label
0,many members of the lgbtq community are not op...,0
1,social news about as college costs rise gen z ...,1
2,malaysias health ministry called on thursday f...,0
3,generosity and kindness are qualities that can...,0
4,a hour hearing broadcast on the courts website...,0
